# 02 — Jointure timestamps (Zilinskas ⨝ HF votes)

**Objectif** : enrichir `battles_bt_styled.parquet` avec les dates de vote HF.

**Pourquoi** : le parquet Zilinskas n'a pas de `timestamp` → bloque R1/R2bis/R4.

**Source timestamps** : `ministere-culture/comparia-votes` (streaming, ~149k lignes, ~1 min).

**Output** :
- `data/raw/votes_timestamps.parquet` (cache)
- `data/interim/battles_with_dates.parquet`

In [ ]:
from pathlib import Path
import sys

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))

import matplotlib.pyplot as plt
import pandas as pd

from compariawatch.data import (
    fetch_vote_timestamps,
    join_battles_with_dates,
    load_battles_zilinskas,
)

CACHE  = ROOT / 'data' / 'raw' / 'votes_timestamps.parquet'
OUTPUT = ROOT / 'data' / 'interim' / 'battles_with_dates.parquet'
FIGURES = ROOT / 'paper' / 'figures'

print('Imports OK')

In [ ]:
# Cell 2 — Smoke test (10 battles, 100 votes streamés)
# Décommenter pour valider avant le run complet
# battles_smoke = load_battles_zilinskas(ROOT).head(10)
# ts_smoke = fetch_vote_timestamps(limit=100)
# join_battles_with_dates(battles_smoke, ts_smoke).head()

In [ ]:
# Cell 3 — Run complet (utilise le cache si déjà téléchargé)
battles = load_battles_zilinskas(ROOT)
print(f'Battles Zilinskas : {len(battles):,} lignes')

timestamps = fetch_vote_timestamps(cache_path=CACHE)
print(f'Timestamps votes  : {len(timestamps):,} paires uniques')

df = join_battles_with_dates(battles, timestamps)

OUTPUT.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(OUTPUT, index=False)
print(f'Sauvegardé : {OUTPUT}')

In [ ]:
# Cell 4 — Qualité jointure
n = len(df)
matched = df['timestamp'].notna().sum()
print(f'Match rate : {matched:,}/{n:,} ({100*matched/n:.1f}%)')
print(f'Période    : {df["date"].min()} → {df["date"].max()}')
print(f'Cohortes   : {df["month"].nunique()} mois')

# Battles sans timestamp = surtout source "reaction" (pas dans votes)
if 'source' in df.columns:
    miss = df[df['timestamp'].isna()]
    print('\nSans timestamp par source :')
    print(miss['source'].value_counts().to_string())

In [ ]:
# Cell 5 — Distribution mensuelle (input R1/R2bis/R4)
monthly = df.dropna(subset=['month']).groupby('month').size().sort_index()
print('Battles par mois :')
print(monthly.to_string())

fig, ax = plt.subplots(figsize=(12, 4))
monthly.plot(kind='bar', ax=ax, color='steelblue', alpha=0.8)
ax.set_xlabel('Mois')
ax.set_ylabel('Nb battles')
ax.set_title(f'Volume de battles par cohorte mensuelle (N={len(monthly)} mois)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(FIGURES / 'EDA_battles_monthly.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure : paper/figures/EDA_battles_monthly.png')

In [ ]:
# Cell 6 — Faisabilité R1 : modèles par cohorte
dec = df[df['winner'].isin(['model_a', 'model_b'])].dropna(subset=['month'])
models_per_month = (
    dec.groupby('month')
    .apply(lambda g: len(set(g['model_a_name']) | set(g['model_b_name'])), include_groups=False)
)
print('Modèles uniques par mois (seuil R1 : ≥ 5) :')
print(models_per_month.to_string())
ok = (models_per_month >= 5).sum()
print(f'\nCohortes exploitables (≥5 modèles) : {ok}/{len(models_per_month)}')